In [ ]:
from config.config import config



In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy import String, Integer, DateTime

user = config["mariadb"]["user"]
password = config["mariadb"]["password"]
ip = config["mariadb"]["ip"]
port = config["mariadb"]["port"]
db = config["mariadb"]["db"]
url = f"mysql+pymysql://{user}:{password}@{ip}:{port}/{db}"

engine = create_engine(url, echo=True)

class Base(DeclarativeBase):
    pass

class Users(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(Integer, comment="user id", primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(200), comment="real name of user")
    email: Mapped[str] = mapped_column(String(200), comment="email address of user")
    age: Mapped[int] = mapped_column(Integer, comment="age of user")

class Plants(Base):
    __tablename__ = "plants"
    id: Mapped[int] = mapped_column(Integer, comment="plant id", primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(200), comment="nickname of plant")
    birthday: Mapped[str] = mapped_column(DateTime, comment="birthday of plant", nullable=True)
    user_id: Mapped[int] = mapped_column(Integer, comment="owner of plant")

# Base.metadata.create_all(bind=engine)

In [ ]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)

with Session() as s:
    user = Users(name="Pengoo", email="pengoo@pengpeng.com", age=10)
    plant = Plants(name="EucalyPengoo", user_id=0)
    s.add(user)
    s.add(plant)
    s.commit()

In [ ]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)

names  = ["pingoo", "edward", "elen", "iverson"]
emails = ["pingoo@example.com", "ed@ex.com", "elen@ex.com", "shoot@76s.com"]
ages   = [8, 23, 56, 20]

with Session() as s:
    for name, email, age in zip(names, emails, ages):
        user = Users(name=name, email=email, age=age)
        s.add(user)    
    s.commit()

In [ ]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)

with Session() as s:
    result = s.get(Users, 1)

print(result.id, result.name, result.email, result.age)

In [ ]:
from sqlalchemy import select

Session = sessionmaker(bind=engine)

with Session() as s:
    result = select(Users).where(Users.age < 20)
    users = s.scalars(result).all()
    
    for user in users:
        print(user.id, user.name, user.email, user.age)

In [ ]:
from sqlalchemy import select

Session = sessionmaker(bind=engine)

with Session() as s:
    # with get
    pingoo = s.get(Users, 2)
    pingoo.email = "pingping@ex.com"
    # with select
    result = select(Users).where(Users.name == "pengoo")
    pengoo = s.scalar(result)
    pengoo.name = "pengoo"
    pengoo.email = "king-pengoo@ex.com"
    # commit
    s.commit()
    
    # retrieve
    result = select(Users).where(Users.age < 20)
    users = s.scalars(result).all()
    for user in users:
        print(user.id, user.name, user.email, user.age)

In [ ]:
from sqlalchemy import select
from sqlalchemy import create_engine

user = config["mariadb"]["user"]
password = config["mariadb"]["password"]
ip = config["mariadb"]["ip"]
port = config["mariadb"]["port"]
db = config["mariadb"]["db"]
url = f"mysql+pymysql://{user}:{password}@{ip}:{port}/{db}"

engine = create_engine(url, echo=True)
Session = sessionmaker(bind=engine)

with Session() as s:
    # edward 를 삭제
    result = select(Users).where(Users.name == "edward")
    ed = s.scalar(result)
    s.delete(ed)
    s.commit()

In [ ]:
from sqlalchemy import select

Session = sessionmaker(bind=engine)

with Session() as s:
    result = select(Users)
    users = s.scalars(result)
    for u in users:
        print(u.id, u.name, u.email, u.age)

In [ ]:
from sqlalchemy import select

Session = sessionmaker(bind=engine)

with Session() as s:
    result = select(Users)
    users = s.scalars(result)
    print(users.all()[0].__dict__)

In [ ]:
from sqlalchemy import select

Session = sessionmaker(bind=engine)

with Session() as s:
    result = (
        select(Plants, Users) # 조회할 데이터 (여기선 SELECT Plants.*, Users.*)
        .join(Users,          # 조인할 테이블
              Plants.user_id == Users.id) # JOIN 조건
    )
    rows = s.execute(result).all()
    for plant, user in rows:
        print(user.name, plant.name)

In [ ]:
from sqlalchemy import select

Session = sessionmaker(bind=engine)

with Session() as s:
    result = select(Users.name, Users.age)
    users = s.execute(result).all()
    for u in users:
        print(u.name, u.age)

In [ ]:
from sqlalchemy import select
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)

with Session() as s:
    result = (
        select(Users.name.label("user_name"),
               Plants.name.label("plant_name")) # 조회할 데이터 (여기선 SELECT Plants.*, Users.*)
        .join(Plants,                     # 조인할 테이블
              Users.id == Plants.user_id, # JOIN 조건
              isouter=True)               # OUTER JOIN
    )
    rows = s.execute(result).all()
    for u, p in rows:
        print(u, p)

In [ ]:
from sqlalchemy import select
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)

with Session() as s:
    result = select(Users.name.label("who"),
                    Users.age.label("years_from_birth"))
    rows = s.execute(result).all()
    for a, b in rows:
        print(a, b) # 별칭으로 꺼내 씀

In [ ]:
from sqlalchemy import select
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)

with Session() as s:
    result = (
        select(Users.name.label("user_name"),
               Plants.name.label("plant_name"))
        .join(Plants,                    
              Users.id == Plants.user_id,
              full=True)               
    )
    rows = s.execute(result).all()
    for u, p in rows:
        print(u, p)